### Cell 1：导入模块和控件

In [1]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import sympy as sp

# 从我们的模块导入
from equations import f, df, phi, generate_matrix
from nonlinear_solvers import (
    simple_iteration,
    steffensen,
    newton,
    newton_downhill
)
from eigen_solvers import power_method, inverse_power_method

### Cell 2：非线性方程求根控件

In [2]:
# 公共参数控件
x0_slider = widgets.FloatText(value=1.0, min=0.0, max=2.0, step=0.01, description='初值 x0:')
eps_text = widgets.BoundedFloatText(value=1e-6, min=1e-15, max=1e-1, step=1e-7, description='误差限:', format='.1e')
max_iter_slider = widgets.IntText(value=100, min=10, max=500, description='最大迭代次数:')

# 牛顿下山法专用控件
lambda_eps = widgets.BoundedFloatText(value=1e-8, min=1e-15, max=1e-1, description='下山下限:', format='.1e')
max_downhill_slider = widgets.IntText(value=20, min=5, max=50, description='最大下山次数:')

# 方法选择
method_selector = widgets.Dropdown(
    options=['简单迭代法', 'Steffensen法', '牛顿迭代法', '牛顿下山法'],
    value='简单迭代法',
    description='方法:'
)

run_button = widgets.Button(description='开始求根', button_style='success')
output_area = widgets.Output()

# ==================== 方程自定义控件 ====================
equation_mode = widgets.RadioButtons(
    options=['默认方程', '自定义方程'],
    value='默认方程',
    description='方程来源:'
)

# 自定义方程输入框
f_expr_text = widgets.Text(
    value='x**3 + 2*x**2 + 10*x - 20',
    description='f(x)=',
    layout=widgets.Layout(width='400px')
)
phi_expr_text = widgets.Text(
    value='20 / (x**2 + 2*x + 10)',
    description='φ(x)=',
    layout=widgets.Layout(width='400px')
)
df_expr_text = widgets.Text(
    value='',  # 留空则自动求导
    placeholder='留空则自动求导',
    description='f\'(x)=',
    layout=widgets.Layout(width='400px')
)

parse_eq_button = widgets.Button(description='解析并更新方程', button_style='warning')
eq_status = widgets.Output()

# 存储当前使用的函数（默认指向 equations.py 中的函数）
current_f = f
current_df = df
current_phi = phi

def update_eq_status():
    with eq_status:
        clear_output()
        print("当前使用方程：")
        print(f"  f(x) = {f_expr_text.value}")
        print(f"  φ(x) = {phi_expr_text.value}")
        if df_expr_text.value.strip():
            print(f"  f'(x) = {df_expr_text.value.strip()}")
        else:
            print("  f'(x) = 自动求导")

def on_parse_eq(b):
    global current_f, current_df, current_phi
    x = sp.Symbol('x')
    try:
        # 解析 f(x)
        f_expr = sp.sympify(f_expr_text.value)
        f_func = sp.lambdify(x, f_expr, 'numpy')

        # 解析或自动求导 f'(x)
        if df_expr_text.value.strip():
            df_expr = sp.sympify(df_expr_text.value)
            df_func = sp.lambdify(x, df_expr, 'numpy')
        else:
            df_expr = sp.diff(f_expr, x)
            df_func = sp.lambdify(x, df_expr, 'numpy')
            # 更新文本框显示自动求导结果（可选）
            df_expr_text.value = str(df_expr)

        # 解析 φ(x)
        phi_expr = sp.sympify(phi_expr_text.value)
        phi_func = sp.lambdify(x, phi_expr, 'numpy')

        # 更新全局函数
        current_f = f_func
        current_df = df_func
        current_phi = phi_func

        with eq_status:
            clear_output()
            print("✅ 方程解析成功！")
            print(f"  f(x) = {f_expr}")
            print(f"  φ(x) = {phi_expr}")
            print(f"  f'(x) = {df_expr}")
    except Exception as e:
        with eq_status:
            clear_output()
            print(f"❌ 解析失败: {e}")

def on_eq_mode_change(change):
    if change['new'] == '默认方程':
        # 切回默认函数
        global current_f, current_df, current_phi
        from equations import f as default_f, df as default_df, phi as default_phi
        current_f = default_f
        current_df = default_df
        current_phi = default_phi
        # 隐藏自定义输入框
        f_expr_text.layout.visibility = 'hidden'
        phi_expr_text.layout.visibility = 'hidden'
        df_expr_text.layout.visibility = 'hidden'
        parse_eq_button.layout.visibility = 'hidden'
        with eq_status:
            clear_output()
            print("当前使用默认方程：f(x) = x^3 + 2x^2 + 10x - 20")
    else:
        f_expr_text.layout.visibility = 'visible'
        phi_expr_text.layout.visibility = 'visible'
        df_expr_text.layout.visibility = 'visible'
        parse_eq_button.layout.visibility = 'visible'
        update_eq_status()

equation_mode.observe(on_eq_mode_change, names='value')
parse_eq_button.on_click(on_parse_eq)

# 界面组合
equation_box = widgets.VBox([
    equation_mode,
    widgets.HBox([f_expr_text, phi_expr_text]),
    df_expr_text,
    parse_eq_button,
    eq_status
])

# 初始状态（默认方程模式）
f_expr_text.layout.visibility = 'hidden'
phi_expr_text.layout.visibility = 'hidden'
df_expr_text.layout.visibility = 'hidden'
parse_eq_button.layout.visibility = 'hidden'

### Cell 3：求根事件响应函数

In [3]:
def on_run_clicked(b):
    with output_area:
        clear_output()
        x0 = x0_slider.value
        eps = eps_text.value
        max_iter = max_iter_slider.value
        method = method_selector.value

        # 使用当前激活的方程函数
        f_now = current_f
        df_now = current_df
        phi_now = current_phi

        try:
            if method == '简单迭代法':
                root, iters, conv = simple_iteration(phi_now, x0, eps, max_iter)
                print(f"【{method}】")
                print(f"近似根: {root:.10f}")
                print(f"迭代次数: {iters}, 收敛情况: {'收敛' if conv else '未收敛'}")
                print(f"f(root) = {f_now(root):.2e}")
            elif method == 'Steffensen法':
                root, iters, conv = steffensen(phi_now, x0, eps, max_iter)
                print(f"【{method}】")
                print(f"近似根: {root:.10f}")
                print(f"迭代次数: {iters}, 收敛情况: {'收敛' if conv else '未收敛'}")
                print(f"f(root) = {f_now(root):.2e}")
            elif method == '牛顿迭代法':
                root, iters, conv = newton(f_now, df_now, x0, eps, max_iter)
                print(f"【{method}】")
                print(f"近似根: {root:.10f}")
                print(f"迭代次数: {iters}, 收敛情况: {'收敛' if conv else '未收敛'}")
                print(f"f(root) = {f_now(root):.2e}")
            elif method == '牛顿下山法':
                root, iters, lambdas, conv = newton_downhill(
                    f_now, df_now, x0, eps, max_iter,
                    epsilon_lambda=lambda_eps.value,
                    max_downhill=max_downhill_slider.value
                )
                print(f"【{method}】")
                print(f"近似根: {root:.10f}")
                print(f"迭代次数: {iters}, 收敛情况: {'收敛' if conv else '未收敛'}")
                print(f"f(root) = {f_now(root):.2e}")
                print(f"各步下山因子: {np.round(lambdas, 6).tolist()}")
        except Exception as e:
            print(f"计算出错：{e}")

run_button.on_click(on_run_clicked)

### Cell 4：非线性方程求根界面布局

In [4]:
nonlinear_box = widgets.VBox([
    equation_box,
    widgets.HBox([x0_slider, eps_text]),
    widgets.HBox([max_iter_slider, method_selector]),
    widgets.HBox([lambda_eps, max_downhill_slider]),
    run_button,
    output_area
])
display(nonlinear_box)

### Cell 5：矩阵特征值控件

In [6]:
# 矩阵输入方式选择
input_mode = widgets.RadioButtons(
    options=['随机生成', '手动输入'],
    value='随机生成',
    description='矩阵来源:'
)

# 随机生成控件
size_input = widgets.IntText(value=4, description='矩阵大小:')
seed_text = widgets.IntText(value=42, description='随机种子:')
regenerate_button = widgets.Button(description='生成随机矩阵')

# 手动输入控件
matrix_textarea = widgets.Textarea(
    value='1 2 3 4;\n5 6 7 8;\n9 10 11 12;\n13 14 15 16',
    placeholder='输入矩阵，例如：1 2;3 4 或 1,2,3;4,5,6',
    description='矩阵:',
    layout=widgets.Layout(width='400px', height='100px')
)
parse_button = widgets.Button(description='解析矩阵', button_style='info')

# 特征值方法选择
eigen_method_selector = widgets.Dropdown(
    options=['幂法', '逆幂法'],
    value='幂法',
    description='方法:'
)
run_eigen_button = widgets.Button(description='开始计算特征值', button_style='success')
eigen_output = widgets.Output()

# 当前矩阵变量
current_A = generate_matrix(4, 42)
matrix_display = widgets.Output()

### Cell 6：矩阵生成与事件函数

In [9]:
def update_matrix_display():
    with matrix_display:
        clear_output()
        print("当前矩阵 A =")
        print(np.round(current_A, 4))

def on_regenerate(b):
    global current_A
    if seed_text.value > 0:
        current_A = generate_matrix(size_input.value, seed_text.value)
    else:
        current_A = generate_matrix(size_input.value)
    update_matrix_display()

def parse_matrix_from_text(text):
    """从字符串解析矩阵，支持空格、逗号、分号、换行分隔"""
    # 按分号或换行切分行
    rows = text.replace(';', '\n').strip().split('\n')
    data = []
    for row in rows:
        row = row.strip()
        if not row:
            continue
        # 用逗号或空格分割
        parts = row.replace(',', ' ').split()
        if not parts:
            continue
        # 转换为浮点数
        numbers = [float(x) for x in parts]
        data.append(numbers)
    if not data:
        raise ValueError("矩阵为空，请检查输入格式")
    # 检查每行列数一致
    ncol = len(data[0])
    for i, row in enumerate(data):
        if len(row) != ncol:
            raise ValueError(f"第 {i+1} 行元素个数不匹配（应为 {ncol}，实际 {len(row)}）")
    return np.array(data)

def on_parse(b):
    global current_A
    with eigen_output:
        clear_output()
        try:
            current_A = parse_matrix_from_text(matrix_textarea.value)
            # 方阵检查
            if current_A.shape[0] != current_A.shape[1]:
                print("矩阵不是方阵（行数 ≠ 列数），特征值方法需要方阵。")
                return
            update_matrix_display()
            print("矩阵解析成功，已更新。")
        except Exception as e:
            print(f"解析失败：{e}")

def on_mode_change(change):
    # 根据选择显示/隐藏相关控件
    if change['new'] == '随机生成':
        size_input.layout.visibility = 'visible'
        seed_text.layout.visibility = 'visible'
        regenerate_button.layout.visibility = 'visible'
        matrix_textarea.layout.visibility = 'hidden'
        parse_button.layout.visibility = 'hidden'
    else:
        size_input.layout.visibility = 'hidden'
        seed_text.layout.visibility = 'hidden'
        regenerate_button.layout.visibility = 'hidden'
        matrix_textarea.layout.visibility = 'visible'
        parse_button.layout.visibility = 'visible'

input_mode.observe(on_mode_change, names='value')

def on_run_eigen(b):
    with eigen_output:
        clear_output()
        A = current_A
        eps = eps_text.value
        max_iter = max_iter_slider.value
        method = eigen_method_selector.value

        try:
            if method == '幂法':
                val, vec, iters, conv = power_method(A, epsilon=eps, max_iter=max_iter)
                print(f"【幂法：主特征值】")
            else:
                # 尝试求逆，若矩阵奇异则给出提示
                if np.linalg.matrix_rank(A) < A.shape[0]:
                    print("矩阵奇异，无法使用逆幂法（需要可逆矩阵）")
                    return
                val, vec, iters, conv = inverse_power_method(A, epsilon=eps, max_iter=max_iter)
                print(f"【逆幂法：最小特征值】")
            print(f"特征值: {val:.10f}")
            print(f"迭代次数: {iters}, 收敛: {'收敛' if conv else '未收敛'}")
            print(f"特征向量: {np.round(vec, 6)}")

            # 矩阵条件数与残差检查（检查矩阵是否非奇异）
            cond = np.linalg.cond(A)
            if cond > 1e10:
                print(f"⚠️ 矩阵条件数 = {cond:.2e}（很大），结果可能不可靠")

            residual = np.linalg.norm(A @ vec - val * vec)
            print(f"残差 ||A·vec - λ·vec|| = {residual:.2e}")
            if residual > 1e-4:
                print("⚠️ 残差较大，检查特征值/向量是否正确")

            # 对比 numpy 真实特征值
            eigvals, _ = np.linalg.eig(A)
            print(f"\nnumpy 全部特征值: {np.round(eigvals, 6)}")
            if method == '幂法':
                idx = np.argmax(np.abs(eigvals))
            else:
                idx = np.argmin(np.abs(eigvals))
            print(f"参考值: {eigvals[idx]:.6f}")
        except np.linalg.LinAlgError:
            print("矩阵奇异，无法求逆（逆幂法需要可逆矩阵）")
        except Exception as e:
            print(f"计算错误：{e}")

# 绑定按钮事件
regenerate_button.on_click(on_regenerate)
parse_button.on_click(on_parse)
run_eigen_button.on_click(on_run_eigen)

# 初始显示
update_matrix_display()
# 默认随机生成模式，手动输入控件隐藏
matrix_textarea.layout.visibility = 'hidden'
parse_button.layout.visibility = 'hidden'

### Cell 7：特征值界面布局

In [12]:
eigen_box = widgets.VBox([
    input_mode,
    widgets.HBox([size_input, seed_text, regenerate_button]),
    widgets.HBox([matrix_textarea, parse_button]),
    matrix_display,
    widgets.HBox([eigen_method_selector, run_eigen_button]),
    eigen_output
])
display(eigen_box)